# 3. GOTOBI EFFECT · ROBUSTNESS BATTERY & VALIDATION GATE

Robustness battery of the **USDJPY Gotobi** rule on M5 Axi bars (UTC), following the structure of notebook 03 of the Monday / RSI2 edges. The rules are the ones frozen in notebooks 01 and 02 (`variants/*.yaml`): **nothing is re-tuned here**.

### BATTERY
1. **Out-of-sample**: untouched data since `splits.oos_start` (gates: `n_oos_min`, `t_oos_min`, `alpha`, `pf_min`).
2. **Walk-forward** with fixed rules (3 y IS -> 1 y OOS).
3. **Statistical evidence (edge-specific)**: Bonferroni threshold, **placebo calendars** (dates shifted by k business days) and **permutation test** against random non-Gotobi days.
4. **Costs across periods**: the feed spread is a modelled constant before ~2020, so the historical net result is checked with flat spreads; breakeven multiple.
5. **Hard limit & sizing**: intrabar drawdown vs `risk.hard_limit`, maximum fraction, annual table.
6. **Block Monte Carlo** on the 5-minute MTM paths of every trade.
7. **Stress & tail**: costs x2, best years / best trades removed, season split, opt-in stop test.
8. **Edge decay**: rolling mean / PF, recent vs historical, z wear-and-tear protocol.
9. **PASS / FAIL registry** (`Checks`, same as notebook 02) and 10. **save** for notebook 04.

In [ ]:
VARIANT = "v1_usdjpy"     # variants/<name>.yaml
MC_BLOCK = 5              # block length (trades) of the Monte Carlo
MC_MAX_BREACH = 0.05      # max simulated probability of touching the hard limit
STOPS_BPS = [0, 10, 20, 30, 50]   # opt-in stop levels for the tail test (the rule itself has no stop)

## 1. IMPORTS AND CONFIG

In [ ]:
import sys, importlib
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

def find_upwards(name, start=None):
    """First parent folder (cwd included) that contains the folder `name`."""
    here = (start or Path.cwd()).resolve()
    for base in (here, *here.parents):
        if (base / name).is_dir():
            return base
    raise FileNotFoundError(f"Could not find '{name}' walking up from {here}")

EDGE_DIR = find_upwards("variants")
if str(EDGE_DIR) not in sys.path:
    sys.path.insert(0, str(EDGE_DIR))

import gotobi_engine as eng
import gotobi_utils as U
import variant_config
from variant_config import load_variant, save_json, load_json

importlib.reload(eng); importlib.reload(U)
variant_config.EDGE_DIR = EDGE_DIR

cfg = load_variant(VARIANT)
eng.set_engine_defaults(**cfg["engine_defaults"])
INS, RULE, COSTS, ACC, RISK = cfg["instrument"], cfg["rule"], cfg["costs"], cfg["acceptance"], cfg["risk"]
OOS_START, RUN_DIR = cfg["oos_start_ts"], cfg["run_dir"]
ET, XT = RULE["entry_time_jst"], RULE["exit_time_jst"]
MODE, SCALE = COSTS["cost_mode"], float(COSTS["spread_scale"])
FLAT = sorted({float(x) for x in np.atleast_1d(COSTS["robustness"]["flat_spread_pips"])})
PF_MIN = float(ACC["pf_min"])
ALPHA, T_OOS_MIN, N_OOS_MIN = float(ACC["alpha"]), float(ACC["t_oos_min"]), int(ACC["n_oos_min"])
DRAWS, SEED, ROLL = int(ACC["perm_draws"]), int(ACC["seed"]), int(ACC["roll_trades"])
HARD, FRAC = float(RISK["hard_limit"]), float(RISK["size_fraction"])
MIN_T = float(cfg["evidence_thresholds"]["min_t_net"])
K_HYP = int(cfg["evidence_thresholds"]["hypotheses_tested"])
PLACEBO = [int(k) for k in cfg["placebo_offsets_bdays"]]
share = lambda x: x / 100 if x > 1 else float(x)
C = U.Checks()

print(f"Variant {cfg['id']} | {INS['symbol']} ({INS['broker']}) | long {ET} -> {XT} JST | cost_mode={MODE}, spread_scale={SCALE:g}")
print(f"Risk: hard limit -{HARD:.0%}, nominal fraction {FRAC:.0%} | OOS from {OOS_START:%Y-%m-%d} | rules hash {variant_config.rules_hash(cfg)} | run folder {RUN_DIR}")

# ---- helpers (same as notebook 02) ----
def summ(t):
    s = eng.trade_stats(t)
    return dict(n=s["n"], pf=s["pf"], mean=s["mean_bps"], t=s["t"], win=100 * s["win_rate"])

def parts_table(parts):
    out = pd.DataFrame({k: summ(t) for k, t in parts.items()}).T.astype(float)
    out["se"] = [U.mean_se(t["pnl_bps"])[1] for t in parts.values()]
    return out

def bars(ax, tbl, title):
    x = np.arange(len(tbl))
    ax.bar(x, tbl["mean"], yerr=tbl["se"], width=0.6, ecolor=U.SUB, color=[U.GREEN if v > 0 else U.RED for v in tbl["mean"]])
    ax.axhline(0, color=U.SUB, lw=0.6)
    ax.set_xticks(x); ax.set_xticklabels([str(i) for i in tbl.index], color=U.SUB)
    for i, n in enumerate(tbl["n"]):
        ax.text(i, ax.get_ylim()[0], f"N={int(n)}", color=U.SUB, fontsize=8, ha="center", va="bottom")
    U.set_title(ax, title)

def heatmap(ax, g, title, center=0.0, fmt="{:.1f}"):
    v = g.to_numpy(float)
    span = np.nanmax(np.abs(v - center)) or 1.0
    ax.imshow(v, cmap="RdYlGn", vmin=center - span, vmax=center + span, aspect="auto")
    ax.set_xticks(range(g.shape[1])); ax.set_xticklabels(g.columns, rotation=30, ha="right", color=U.SUB)
    ax.set_yticks(range(g.shape[0])); ax.set_yticklabels(g.index, color=U.SUB)
    for (r, c), val in np.ndenumerate(v):
        ax.text(c, r, fmt.format(val), ha="center", va="center", fontsize=8, color="#15171c")
    U.set_title(ax, title)

def breakeven(tbl):
    z = U.interp_zero(tbl.index, tbl["mean"])
    return np.inf if np.isnan(z) and (tbl["mean"] > 0).all() else z

## 2. OUT-OF-SAMPLE PERFORMANCE

The engine has no warm-up, so the rules run once on the full history and trades are split by JST entry date (`oos_start` is a JST midnight: no position crosses it). Returns are mark-to-market per business day at the nominal fraction `risk.size_fraction`. Gates: `n_oos_min`, one-sided t and p (`t_oos_min`, `alpha`) and `pf_min`.

In [ ]:
df_raw = eng.load_data(start=INS["start"], end=INS.get("end"))
a, b = (df_raw.index[0] + eng.JST).normalize(), (df_raw.index[-1] + eng.JST).normalize()
trades = eng.run_gotobi_backtest(df_raw)
ret = eng.daily_returns(df_raw, trades, FRAC)
ent = pd.to_datetime(trades["entry_date"])
SPLIT = {"In-sample": (a, OOS_START - pd.Timedelta(days=1)), "Out-of-sample": (OOS_START, b), "Full": (a, b)}
sub = lambda t, lo, hi: t[(pd.to_datetime(t["entry_date"]) >= lo) & (pd.to_datetime(t["entry_date"]) <= hi)]

rows = {}
for k, (lo, hi) in SPLIT.items():
    r, yrs = ret.loc[lo:hi], (hi - lo).days / 365.25
    eq = (1 + r).cumprod()
    rows[k] = {**summ(sub(trades, lo, hi)), "ret_%": 100 * (eq.iloc[-1] - 1), "cagr_%": 100 * (eq.iloc[-1] ** (1 / yrs) - 1),
               "sharpe": r.mean() / r.std() * np.sqrt(252), "maxdd_close_%": 100 * U.drawdown(r).min()}
perf = pd.DataFrame(rows).T
t_is, t_oos = sub(trades, *SPLIT["In-sample"]), sub(trades, *SPLIT["Out-of-sample"])
S_is, S_oos, S_all = summ(t_is), summ(t_oos), summ(trades)
st_oos = eng.trade_stats(t_oos)

print("=" * 80)
print(f"PERFORMANCE · net of costs · return metrics at {FRAC:.0%} nominal fraction")
print("=" * 80)
display(perf.round(3))
print(f"Mean net IS {S_is['mean']:+.2f} bps -> OOS {S_oos['mean']:+.2f} bps (retention {S_oos['mean'] / S_is['mean']:.2f}) | OOS t {S_oos['t']:.2f}, one-sided p {st_oos['p_one']:.3f} | OOS N {S_oos['n']}")

fig, (ax1, ax2) = U.subplots(2, 1, figsize=(14, 6.5), sharex=True, gridspec_kw=dict(height_ratios=[3, 1]))
ax1.plot(ret.index, (1 + ret).cumprod(), color=U.BLUE, lw=1.5, label=f"{FRAC:.0%} nominal fraction")
ax2.fill_between(ret.index, 100 * U.drawdown(ret), 0, color=U.RED, alpha=0.5)
ax2.axhline(-100 * HARD, color=U.ORANGE, ls="--", lw=1.2, label=f"hard limit -{HARD:.0%}")
for ax in (ax1, ax2):
    ax.axvline(OOS_START, color=U.ORANGE, ls="--", lw=1.2)
ax1.set_yscale("log"); ax2.set_ylabel("Daily-close DD %", color=U.SUB)
U.set_title(ax1, f"Equity curve (log) · dashed = OOS start {OOS_START:%Y-%m-%d}"); U.legend(ax1); U.legend(ax2, "lower left")
plt.tight_layout(); plt.show()

C("2 OOS", "N OOS >= n_oos_min", S_oos["n"] >= N_OOS_MIN, f"{S_oos['n']} vs {N_OOS_MIN}")
C("2 OOS", "t net OOS >= t_oos_min", S_oos["t"] >= T_OOS_MIN, f"{S_oos['t']:.2f} vs {T_OOS_MIN:g}")
C("2 OOS", "one-sided p OOS < alpha", st_oos["p_one"] < ALPHA, f"p {st_oos['p_one']:.3f} vs {ALPHA:g}")
C("2 OOS", "PF net OOS >= pf_min", S_oos["pf"] >= PF_MIN, f"{S_oos['pf']:.3f} vs {PF_MIN}")

## 3. WALK-FORWARD (FIXED RULES, 3 y IS -> 1 y OOS)

No refitting: each calendar year is compared with the annualised return of the three full years before it. Gates: efficiency (mean OOS / mean IS) >= 0.5, positive years >= `years_pos_min`, worst-year drawdown inside the hard limit.

In [ ]:
wf = eng.walk_forward_fixed(df_raw, rules={}, is_years=3, sizing=FRAC)
eff = wf["ret_oos"].mean() / wf["ret_is_annual"].mean() if wf["ret_is_annual"].mean() > 0 else np.nan
pos = float((wf["ret_oos"] > 0).mean())
worst = float(wf["mdd_oos"].min())
print(f"{len(wf)} folds | efficiency {eff:.2f} (>= 0.50) | positive years {100 * pos:.0f}% ({int((wf['ret_oos'] > 0).sum())}/{len(wf)}) | worst-year DD {100 * worst:.2f}% (limit -{HARD:.0%})")
display(wf.round(4))

fig, ax = U.subplots(figsize=(14, 4.5))
x, w = np.arange(len(wf)), 0.38
ax.bar(x - w / 2, 100 * wf["ret_is_annual"], w, color=U.BLUE, alpha=0.8, label="IS annualised (%)")
ax.bar(x + w / 2, 100 * wf["ret_oos"], w, color=[U.GREEN if r > 0 else U.RED for r in wf["ret_oos"]], label="OOS year (%)")
ax.axhline(0, color=U.SUB, lw=0.8); ax.set_xticks(x); ax.set_xticklabels(wf["year"], color=U.SUB)
U.set_title(ax, "Walk-forward by calendar year"); U.legend(ax)
plt.tight_layout(); plt.show()

C("3 Walk-forward", "efficiency >= 0.5", bool(np.isfinite(eff) and eff >= 0.5), f"{eff:.2f}")
C("3 Walk-forward", "positive OOS years >= years_pos_min", pos >= share(ACC["years_pos_min"]) if ACC["years_pos_min"] <= 1 else pos * len(wf) >= ACC["years_pos_min"], f"{100 * pos:.0f}%")
C("3 Walk-forward", "worst-year DD inside hard limit", worst >= -HARD, f"{100 * worst:.2f}%")

## 4. STATISTICAL EVIDENCE: HOW LIKELY IS THIS TO BE CHANCE?

1. **Multiple-testing bar**: `hypothesis_threshold(k)` (Bonferroni, one-sided) with the `hypotheses_tested` of the YAML.
2. **Placebo calendars**: the same window and costs on the Gotobi calendar shifted by k business days. If the effect is real it lives on the true dates and fades on the shifted ones.
3. **Permutation test** (gross window return): Gotobi dates vs random draws of non-Gotobi business days; once with all of them, once excluding the neighbouring business days, and once on the OOS slice alone.

In [ ]:
t_bonf = eng.hypothesis_threshold(K_HYP)
print(f"Net t (full) {S_all['t']:.2f} | min_t_net {MIN_T:g} | Bonferroni threshold ({K_HYP} hypotheses) {t_bonf:.2f}")

plc = eng.placebo_table(df_raw, offsets=PLACEBO)
real, others = plc.loc[0], plc.drop(index=0)
fig, (ax1, ax2) = U.subplots(1, 2, figsize=(14, 4.2), grid=False)
for ax, col, name in ((ax1, "gross", "Gross"), (ax2, "net", "Net")):
    v, tt = plc[f"{col}_bps"], plc[f"{col}_t"]
    ax.bar([f"{k:+d}" for k in plc.index], v, color=[U.BLUE if k == 0 else U.SUB for k in plc.index], width=0.6)
    ax.axhline(0, color=U.SUB, lw=0.6)
    for i, (val, t_) in enumerate(zip(v, tt)):
        ax.text(i, val, f"t={t_:.1f}", color=U.TXT, fontsize=8, ha="center", va="bottom" if val >= 0 else "top")
    ax.set_xlabel("calendar shift (business days; 0 = real)", color=U.SUB); U.set_title(ax, f"{name} mean bps per trade")
plt.tight_layout(); plt.show()
display(plc.round(3))

df_oos = eng.split_df(df_raw, OOS_START)[1]
rds = {"Full · all controls": eng.random_date_test(df_raw, n_draws=DRAWS, seed=SEED),
       "Full · controls without neighbours": eng.random_date_test(df_raw, n_draws=DRAWS, seed=SEED, exclude_neighbors=1),
       "OOS only": eng.random_date_test(df_oos, n_draws=DRAWS, seed=SEED)}
fig, axes = U.subplots(1, 3, figsize=(17, 3.8))
for ax, (name, rd) in zip(axes, rds.items()):
    ax.hist(rd["sims"], bins=50, color=U.SUB, alpha=0.8)
    ax.axvline(rd["real_mean_bps"], color=U.GREEN, lw=2, label=f"real {rd['real_mean_bps']:+.2f} bps")
    ax.set_xlabel("mean gross bps of random dates", color=U.SUB); U.legend(ax, "upper left")
    U.set_title(ax, f"{name} · z {rd['z']:.1f} · p {rd['p_one']:.4f}")
plt.tight_layout(); plt.show()
display(pd.DataFrame({k: {m: v for m, v in rd.items() if m != "sims"} for k, rd in rds.items()}).T.round(4))

C("4 Evidence", "net t (full) >= min_t_net", S_all["t"] >= MIN_T, f"{S_all['t']:.2f} vs {MIN_T:g}")
C("4 Evidence", "net t (full) >= Bonferroni threshold", S_all["t"] >= t_bonf, f"{S_all['t']:.2f} vs {t_bonf:.2f}")
C("4 Evidence", "placebo: real gross bps > every shifted calendar", bool(real["gross_bps"] > others["gross_bps"].max()), f"{real['gross_bps']:+.2f} vs max {others['gross_bps'].max():+.2f}")
C("4 Evidence", "placebo: real net t > every shifted calendar", bool(real["net_t"] > others["net_t"].max()), f"{real['net_t']:.1f} vs max {others['net_t'].max():.1f}")
for name, rd in rds.items():
    C("4 Evidence", f"permutation p < alpha · {name}", rd["p_one"] < ALPHA, f"p {rd['p_one']:.4f}, z {rd['z']:.1f}")

## 5. COSTS ACROSS PERIODS

Until ~2020 the feed spread is a modelled constant, so `spread_scale` is not valid there and the historical net result is read with **flat spreads** (YAML `costs.robustness`). The heatmap shows mean net bps and t by period (YAML `periods`) and cost scenario; sweeps run without the spread filter to isolate the cost. Breakeven multiple = breakeven spread / configured cost.

In [ ]:
NOFILT = dict(max_entry_spread_pips=float("inf"))
SPREADS = sorted({0.0, 0.5, 1.0, 1.5, 2.0, 3.0, 4.0, 6.0, 8.0} | set(FLAT))
SCALES = sorted({0.0, 0.5, 1.0, 1.5, 2.0, 3.0, 4.0, 6.0} | {SCALE, 2 * SCALE})
flat_tr = {}
for s in SPREADS:
    with U.engine_overrides(eng, flat_spread_pips=s):
        flat_tr[s] = eng.run_gotobi_backtest(df_raw, cost_mode="flat_pips", **NOFILT)
feed_tr = {k: eng.run_gotobi_backtest(df_raw, cost_mode="feed", spread_scale=k, **NOFILT) for k in SCALES}
flat_tbl, feed_tbl = parts_table(flat_tr), parts_table(feed_tr)
be_flat, be_scale = breakeven(flat_tbl), breakeven(feed_tbl)
be_mult = (be_scale / SCALE if MODE == "feed" else be_flat / float(eng.ENGINE_DEFAULTS["flat_spread_pips"]) if MODE == "flat_pips" else np.nan)

scen = {"configured": trades, f"feed x{SCALE:g}": feed_tr[SCALE], f"feed x{2 * SCALE:g}": feed_tr[2 * SCALE], **{f"flat {s:g}p": flat_tr[s] for s in FLAT}}
pst = {k: eng.period_stats(t, cfg["periods"]).astype(float) for k, t in scen.items()}
pm = pd.DataFrame({k: v["mean_bps"] for k, v in pst.items()})
pt = pd.DataFrame({k: v["t"] for k, v in pst.items()})
pn = pst["configured"]["n"]

fig, (ax1, ax2) = U.subplots(1, 2, figsize=(16, 4.4), grid=False)
heatmap(ax1, pm, "Mean net bps · period x cost scenario", 0.0)
heatmap(ax2, pt, "t-stat of the mean · period x cost scenario", 0.0)
plt.tight_layout(); plt.show()
print("N per period:", pn.astype(int).to_dict())
display(flat_tbl.round(3))
print(f"Breakeven: {be_flat:.2f} flat pips | spread_scale {be_scale:.2f} (configured {SCALE:g}) | multiple ({MODE}) {be_mult:.2f}x (min {ACC['breakeven_mult_min']})")

worst_flat = f"flat {max(FLAT):g}p"
C("5 Costs", "breakeven multiple >= breakeven_mult_min", None if np.isnan(be_mult) else be_mult >= float(ACC["breakeven_mult_min"]), f"{be_mult:.2f}x")
C("5 Costs", "mean net > 0 (full) at every YAML flat spread", bool((flat_tbl.loc[FLAT, "mean"] > 0).all()), f"worst {flat_tbl.loc[FLAT, 'mean'].min():+.2f} bps")
C("5 Costs", f"periods with mean net > 0 at {worst_flat} >= years_pos_min", float((pm[worst_flat] > 0).mean()) >= share(ACC["years_pos_min"]), f"{int((pm[worst_flat] > 0).sum())}/{len(pm)} periods")

## 6. HARD LIMIT AND SIZING

Long-only, no stop: **the position size is the only risk control**. Drawdown is measured on the equity of the 5-minute bars using each bar's low (intrabar). `max_fraction_for_limit` is the largest fixed nominal fraction (may exceed 100 % = leverage) whose *historical* drawdown still respects the hard limit; it is a ceiling, not a recommendation. This cell runs a bisection over the full M5 history and can take a minute.

In [ ]:
rep = eng.hard_limit_report_intrabar(df_raw, trades, HARD, FRAC)
fmax = eng.max_fraction_for_limit(df_raw, trades, HARD)
annual = eng.annual_returns(ret, trades, HARD)
U.plot_annual_returns(annual, rep, HARD)
display(annual.round(4))
print(f"Intrabar max DD {100 * rep['max_dd']:.2f}% (closes only {100 * rep['max_dd_close']:.2f}%) | limit -{HARD:.0%} | headroom {100 * rep['headroom']:.2f} pp | trough {rep['trough_date']}")
print(f"Max fraction for the limit: {fmax:.2f}x capital | configured {FRAC:.2f}x = {FRAC / fmax:.0%} of the ceiling | years in breach: {int(annual['breach'].sum())}")

C("6 Risk", "historical intrabar max DD inside hard limit", rep["passed"], f"{100 * rep['max_dd']:.2f}% vs -{HARD:.0%}")
C("6 Risk", "no calendar year breaches the hard limit", not bool(annual["breach"].any()), f"{int(annual['breach'].sum())} years")
C("6 Risk", "worst single trade at nominal size < hard limit", bool(trades["pnl_frac"].min() / 100 > -HARD), f"{trades['pnl_frac'].min():.2f}% (pnl_frac)")

## 7. BLOCK MONTE CARLO

Trades are resampled in circular blocks (streaks kept) and every simulated equity path is built from the **5-minute MTM path of each trade** (per trade: growth multiplier, best and worst point inside the trade), so the drawdown is intrabar. Run at the nominal fraction and at the ceiling of section 6, which by construction touches the limit in the real history. Gates: probability of touching the hard limit <= `MC_MAX_BREACH` and p5 of total return > 0.

In [ ]:
def trade_vectors(t, f):
    """Per trade: growth multiplier, highest and lowest equity multiplier inside the trade."""
    g, hi, lo = [], [], []
    for p in eng.trade_mtm_paths(df_raw, t, sizing=f):
        c = np.cumprod(1 + p)
        g.append(c[-1]); hi.append(c.max()); lo.append(c.min())
    return np.array(g), np.array(hi), np.array(lo)

def mc_core(vec, idx):
    """(total return, max intrabar drawdown) of each resampled sequence of trades (rows of idx)."""
    g, hi, lo = vec
    gs = g[idx]
    E = np.cumprod(gs, axis=1); prev = E / gs
    H = np.maximum(prev * hi[idx], E)
    peak = np.concatenate([np.ones((len(idx), 1)), np.maximum(1.0, np.maximum.accumulate(H, axis=1))[:, :-1]], axis=1)
    return E[:, -1] - 1, (prev * lo[idx] / np.maximum(peak, prev) - 1).min(axis=1)

def block_mc(vec, n_sim, block, seed):
    n = len(vec[0])
    rng = np.random.default_rng(seed)
    idx = ((rng.integers(0, n, (n_sim, -(-n // block)))[:, :, None] + np.arange(block)) % n).reshape(n_sim, -1)[:, :n]
    rets, mdds = mc_core(vec, idx)
    real_ret, real_mdd = (x[0] for x in mc_core(vec, np.arange(n)[None, :]))
    return dict(rets=rets, mdds=mdds, ret_p5=np.percentile(rets, 5), ret_p50=np.median(rets), mdd_p50=np.median(mdds),
                mdd_p5=np.percentile(mdds, 5), mdd_worst=mdds.min(), prob_breach=float((mdds <= -HARD).mean()),
                prob_pos=float((rets > 0).mean()), real_mdd=real_mdd, real_ret=real_ret)

MC_SIMS = min(DRAWS, 5000)
SIZES = {f"{FRAC:.2f}x (robot)": FRAC, f"{fmax:.2f}x (ceiling)": fmax}
OFFICIAL = next(iter(SIZES))
mcs = {k: block_mc(trade_vectors(trades, f), MC_SIMS, MC_BLOCK, SEED) for k, f in SIZES.items()}
mc_tbl = pd.DataFrame({k: {"Median return %": 100 * m["ret_p50"], "p5 return %": 100 * m["ret_p5"], "Median max DD %": 100 * m["mdd_p50"],
                           "Worst-5% DD %": 100 * m["mdd_p5"], "Worst simulated DD %": 100 * m["mdd_worst"],
                           f"P(DD beyond -{HARD:.0%}) %": 100 * m["prob_breach"], "P(return > 0) %": 100 * m["prob_pos"],
                           "Real-history DD %": 100 * m["real_mdd"]} for k, m in mcs.items()}).T
display(mc_tbl.round(2))

mc0 = mcs[OFFICIAL]
fig, (ax1, ax2) = U.subplots(1, 2, figsize=(14, 4.4))
ax1.hist(100 * mc0["rets"], bins=60, color=U.BLUE, alpha=0.8); ax1.axvline(0, color=U.RED, lw=1.2); ax1.axvline(100 * mc0["ret_p5"], color=U.ORANGE, ls="--", lw=1.2)
ax2.hist(100 * mc0["mdds"], bins=60, color=U.BLUE, alpha=0.8)
ax2.axvline(-100 * HARD, color=U.RED, lw=1.2, label="hard limit"); ax2.axvline(100 * mc0["mdd_p5"], color=U.ORANGE, ls="--", lw=1.2, label="worst 5%")
ax2.axvline(100 * mc0["real_mdd"], color=U.GREEN, ls=":", lw=1.4, label="real history"); U.legend(ax2, "upper left")
U.set_title(ax1, f"Simulated total return % · {OFFICIAL}"); U.set_title(ax2, "Simulated max intrabar drawdown %")
plt.tight_layout(); plt.show()

C("7 Monte Carlo", "P(touching hard limit) <= max (robot size)", mc0["prob_breach"] <= MC_MAX_BREACH, f"{100 * mc0['prob_breach']:.1f}% vs {100 * MC_MAX_BREACH:.0f}%")
C("7 Monte Carlo", "p5 of total return > 0 (robot size)", mc0["ret_p5"] > 0, f"{100 * mc0['ret_p5']:.1f}%")

## 8. STRESS AND TAIL TESTS

- **Stress table**: costs x2 (without the spread filter), best 2 years removed, best 10 % of trades removed (concentration), winter vs summer entries.
- **Tail / stop test**: the rule has **no stop**; here an opt-in stop (`stop_bps`, filled at min(level, bar open): M5 approximation) shows how much tail it cuts and what it costs. Left: distribution of net PnL; centre: MAE of every trade; right: mean net per stop level.

In [ ]:
if MODE == "feed":
    t_x2 = eng.run_gotobi_backtest(df_raw, spread_scale=2 * SCALE, **NOFILT)
elif MODE == "flat_pips":
    with U.engine_overrides(eng, flat_spread_pips=2 * eng.ENGINE_DEFAULTS["flat_spread_pips"]):
        t_x2 = eng.run_gotobi_backtest(df_raw, **NOFILT)
else:
    t_x2 = eng.run_gotobi_backtest(df_raw, cost_bps_per_side=2 * eng.ENGINE_DEFAULTS["cost_bps_per_side"], **NOFILT)
yr = pd.to_datetime(trades["entry_date"]).dt.year
top2 = trades.groupby(yr)["pnl_bps"].sum().nlargest(2).index.tolist()
k10 = max(1, int(np.ceil(0.10 * len(trades))))
srt = trades["pnl_bps"].sort_values(ascending=False)
share_top10 = float(srt.head(k10).sum() / trades.loc[trades["pnl_bps"] > 0, "pnl_bps"].sum())
winter = ~eng.us_dst_mask(pd.DatetimeIndex(trades["entry_time"]))
stress = parts_table({"Baseline": trades, "Costs x2 (no spread filter)": t_x2, f"Without best 2 years {top2}": trades[~yr.isin(top2)],
                      f"Without best 10% of trades ({k10})": trades.loc[srt.index[k10:]], "Winter entries": trades[winter], "Summer entries": trades[~winter]})
display(stress.round(3))
print(f"Best 10% of trades carry {100 * share_top10:.0f}% of gross profit.")

stop_tr = {("no stop" if s == 0 else f"stop {s} bps"): eng.run_gotobi_backtest(df_raw, stop_bps=float(s)) for s in STOPS_BPS}
stop_tbl = parts_table(stop_tr)
stop_tbl["stopped_%"] = [100 * t["stopped"].mean() for t in stop_tr.values()]
stop_tbl["p1_bps"] = [t["pnl_bps"].quantile(0.01) for t in stop_tr.values()]
stop_tbl["worst_bps"] = [t["pnl_bps"].min() for t in stop_tr.values()]
display(stop_tbl.round(3))

fig, axes = U.subplots(1, 3, figsize=(18, 4.2))
axes[0].hist(trades["pnl_bps"], bins=80, color=U.BLUE, alpha=0.85)
for q, col in ((0.01, U.RED), (0.05, U.ORANGE)):
    axes[0].axvline(trades["pnl_bps"].quantile(q), color=col, ls="--", lw=1.2, label=f"p{int(q * 100)} {trades['pnl_bps'].quantile(q):.1f}")
axes[0].axvline(0, color=U.SUB, lw=0.8); U.legend(axes[0], "upper left"); U.set_title(axes[0], "Net PnL per trade (bps)")
axes[1].hist(trades["mae_bps"], bins=80, color=U.SUB, alpha=0.85)
for s in STOPS_BPS[1:]:
    axes[1].axvline(-s, color=U.ORANGE, ls=":", lw=1)
U.set_title(axes[1], "MAE inside the trade (bps) · dotted = stop levels")
bars(axes[2], stop_tbl, "Mean net bps by stop level")
plt.tight_layout(); plt.show()

C("8 Stress", "costs x2: PF >= 1", stress.loc["Costs x2 (no spread filter)", "pf"] >= 1.0, f"PF {stress.loc['Costs x2 (no spread filter)', 'pf']:.3f}")
C("8 Stress", "without best 2 years: PF >= 1", stress.iloc[2]["pf"] >= 1.0, f"PF {stress.iloc[2]['pf']:.3f}")
C("8 Stress", "without best 10% of trades: mean net >= 0", stress.iloc[3]["mean"] >= 0, f"{stress.iloc[3]['mean']:+.2f} bps")

## 9. EDGE DECAY AND TEMPORAL STABILITY

Rolling mean and PF over `acceptance.roll_trades`, PF by year, recent (last 3 calendar years) vs historical mean, and the **z wear-and-tear protocol**: z = (mean of the last 20 trades - mu0) / (sigma0 / sqrt(20)) with mu0 / sigma0 from the in-sample trades; green > -1, yellow -1 to -2, red <= -2. Inside the IS block it shows how the protocol *would* have warned on the same data.

In [ ]:
pnl = trades.set_index(pd.to_datetime(trades["entry_date"]))["pnl_bps"]
roll_mean, roll_pf = pnl.rolling(ROLL).mean(), pnl.rolling(ROLL).apply(U.profit_factor, raw=True)
by_year = parts_table({int(y): g for y, g in trades.groupby(yr)})
last = int(yr.max())
recent, hist = trades[yr >= last - 2], trades[yr < last - 2]
ratio = recent["pnl_bps"].mean() / hist["pnl_bps"].mean() if hist["pnl_bps"].mean() > 0 else np.nan

W = 20
ref = pnl[pnl.index < OOS_START]
mu0, sd0 = float(ref.mean()), float(ref.std(ddof=1))
z = ((pnl.rolling(W).mean() - mu0) / (sd0 / np.sqrt(W))).dropna()
zone = pd.cut(z, [-np.inf, -2, -1, np.inf], labels=["red", "yellow", "green"])

fig, (ax1, ax2, ax3) = U.subplots(3, 1, figsize=(14, 10), sharex=True)
ax1.plot(roll_mean.index, roll_mean, color=U.GREEN, lw=1.5); ax1.axhline(0, color=U.RED, lw=1.0); ax1.axhline(pnl.mean(), color=U.SUB, ls=":")
ax2.plot(roll_pf.index, roll_pf.clip(upper=6), color=U.BLUE, lw=1.5); ax2.axhline(1, color=U.RED, lw=1.0); ax2.axhline(PF_MIN, color=U.ORANGE, ls="--", lw=1.0)
ax3.plot(z.index, z, color=U.TXT, lw=1.3)
lo_, hi_ = min(z.min() - 0.5, -2.5), max(z.max() + 0.5, 1.5)
ax3.axhspan(-1, hi_, color=U.GREEN, alpha=0.06); ax3.axhspan(-2, -1, color=U.ORANGE, alpha=0.10); ax3.axhspan(lo_, -2, color=U.RED, alpha=0.10); ax3.set_ylim(lo_, hi_)
for ax in (ax1, ax2, ax3):
    ax.axvline(OOS_START, color=U.SUB, ls=":")
U.set_title(ax1, f"Rolling mean net bps ({ROLL} trades)"); U.set_title(ax2, f"Rolling PF ({ROLL} trades, clipped at 6)"); U.set_title(ax3, f"z wear-and-tear protocol (last {W} trades)")
plt.tight_layout(); plt.show()

display(by_year.round(3))
print(f"Recent ({last - 2}-{last}) mean {recent['pnl_bps'].mean():+.2f} bps vs historical {hist['pnl_bps'].mean():+.2f} bps -> ratio {ratio:.2f}")
print(f"Reference (IS): mu0 {mu0:+.2f} bps, sigma0 {sd0:.2f} bps, n {len(ref)} | z today {z.iloc[-1]:+.2f} ({zone.iloc[-1]}) | min z {z.min():+.2f} on {z.idxmin():%Y-%m-%d}")
print("Trades since OOS start by zone:", zone[z.index >= OOS_START].value_counts().to_dict())

C("9 Decay", "recent 3 years / historical mean >= 0.5", bool(np.isfinite(ratio) and ratio >= 0.5), f"{ratio:.2f}")
C("9 Decay", "z today not red (> -2)", bool(z.iloc[-1] > -2), f"z {z.iloc[-1]:+.2f}")
C("9 Decay", "positive years >= years_pos_min (full history)", int((by_year["mean"] > 0).sum()) >= (ACC["years_pos_min"] if ACC["years_pos_min"] > 1 else ACC["years_pos_min"] * len(by_year)), f"{int((by_year['mean'] > 0).sum())}/{len(by_year)}")

## 10. PASS / FAIL SUMMARY, VERDICT AND SAVE

Weaknesses that notebook 02 carried over (failed in-sample checks) are listed next to the verdict: a gate that failed there must be declared here, not silently ignored.

In [ ]:
chk = C.frame()
display(chk.style.map(lambda s: {"PASS": "color:#34d399", "FAIL": "color:#f0555a", "SKIP": "color:#f5a524"}.get(s, ""), subset=["status"]))
cnt = C.counts()
try:
    s02 = load_json(cfg, "02_sensitivity.json")
    weak02 = [c["name"] for c in s02["checks"] if c["status"] == "FAIL"]
except FileNotFoundError:
    s02, weak02 = None, []
ready = cnt["FAIL"] == 0
failed = chk.loc[chk["status"] == "FAIL", "name"].tolist()

print("=" * 80)
print("VERDICT")
print("=" * 80)
print(f"PASS {cnt['PASS']} | FAIL {cnt['FAIL']} | SKIP {cnt['SKIP']}")
print(f"Notebook 02 weaknesses carried over: {weak02 if s02 else 'notebook 02 results not found'}")
print(f"Net {S_all['mean']:+.2f} bps/trade (t {S_all['t']:.1f}) | OOS {S_oos['mean']:+.2f} bps (t {S_oos['t']:.1f}, N {S_oos['n']}) | hard-limit DD {100 * rep['max_dd']:.2f}% at {FRAC:.2f}x (ceiling {fmax:.2f}x) | breakeven {be_mult:.2f}x cost | z {z.iloc[-1]:+.2f}")
print("OVERALL: " + ("PASS. Edge confirmed for the reconciliation stage (notebook 04)." if ready else f"FAIL on {failed}. Do not promote without documenting the failed checks (a margin or an exception must be declared, not silently applied)."))

save_json(cfg, "03_validation.json", dict(
    oos_start=OOS_START.date(), sizing=dict(fraction=FRAC, max_fraction=fmax, hard_limit=HARD),
    performance=perf.to_dict("index"), oos=dict(**S_oos, p_one=st_oos["p_one"]),
    walk_forward=dict(efficiency=eff, pct_positive=pos, worst_mdd=worst, folds=len(wf)),
    evidence=dict(t_bonferroni=t_bonf, placebo=plc.to_dict("index"), permutation={k: {m: v for m, v in rd.items() if m != "sims"} for k, rd in rds.items()}),
    costs=dict(breakeven_flat_pips=be_flat, breakeven_scale=be_scale, breakeven_mult=be_mult, mean_net_by_period=pm.to_dict("index")),
    risk=dict(max_dd=rep["max_dd"], max_dd_close=rep["max_dd_close"], passed=rep["passed"], trough_date=rep["trough_date"], years_in_breach=int(annual["breach"].sum())),
    monte_carlo={k: {m: v for m, v in mc.items() if m not in ("rets", "mdds")} for k, mc in mcs.items()},
    stress=stress.to_dict("index"), decay=dict(recent_ratio=ratio, z_today=float(z.iloc[-1]), mu0=mu0, sd0=sd0),
    carried_from_02=weak02, checks=C.items, counts=cnt, ready_for_04=ready))